##### This notebook shows the process for converting reV distpv CFs to the distpv CF file used in ReEDS
##### To rerun it, the notebook must be copied to the top level of the ReEDS directory

In [ ]:
import pandas as pd
import os
from shapely.geometry import Point
import geopandas as gpd
import numpy as np
import reeds

In [ ]:
inputs_path = '//nrelnas01/ReEDS/FY24-DECARB-KO/distpv_profiles_raw' #TODO: move to publicly accessible location

In [ ]:
def get_rooftop_pv_cf_meta():
    meta = pd.read_csv(os.path.join(inputs_path, "meta.csv"))
    county_centroids = pd.read_csv(os.path.join(inputs_path, "county_centroid_project_points_conus.csv"))
    county_centroids['FIPS'] = 'p' + county_centroids['GEOID'].astype(str).str.zfill(5)
    rooftop_pv_cf_meta = meta.merge(county_centroids[['gid', 'FIPS']], on='gid', how='left')

    rooftop_pv_cf_meta_geometry = [
        Point(xy)
        for xy
        in zip(rooftop_pv_cf_meta['longitude'], rooftop_pv_cf_meta['latitude'])
    ]
    rooftop_pv_cf_meta = gpd.GeoDataFrame(
        rooftop_pv_cf_meta,
        geometry=rooftop_pv_cf_meta_geometry,
        crs='EPSG:4326'
    )

    return rooftop_pv_cf_meta

In [ ]:
def create_gid_county_map(dfcounty, rooftop_pv_cf_meta):
    ## Create mapping between county centroids and rooftop PV profile GIDs
    county_centroids = dfcounty.copy()
    county_centroids['geometry'] = county_centroids['geometry'].centroid

    rooftop_pv_cf_meta_matched = rooftop_pv_cf_meta.loc[(
        rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)
    )]
    
    rooftop_pv_cf_meta_unmatched = rooftop_pv_cf_meta.loc[(
        ~rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)
    )]
    county_centroids_unmatched = county_centroids.loc[(
        ~county_centroids.rb.isin(rooftop_pv_cf_meta_matched.FIPS)
    )]
    rooftop_pv_cf_meta_unmatched = (
        gpd.sjoin_nearest(
            (
                county_centroids_unmatched.drop(columns='FIPS')
                .rename(columns={'rb': 'FIPS'})
                [['FIPS', 'geometry']]
            ),
            (
                rooftop_pv_cf_meta_unmatched.drop(columns='FIPS')
                .to_crs(county_centroids.crs)
            ),
            how='left'
        )
        .to_crs(rooftop_pv_cf_meta.crs)
        .drop(columns='index_right')
    )
    
    gid_county_map = (
        pd.concat([
            rooftop_pv_cf_meta_matched,
            rooftop_pv_cf_meta_unmatched
        ])
        .groupby('gid')
        ['FIPS']
        .apply(list)
        .explode()
    )

    return gid_county_map

In [ ]:
def get_rooftop_pv_cf_profile(sector, rooftop_pv_cf_meta, dfcounty):
    ### Get rooftop PV CF profiles and normalize.
    fpath = os.path.join(inputs_path, f"county_level_distpv_profiles_{sector}_2007_2023.csv")
    rooftop_pv_cf = pd.read_csv(fpath)
    rooftop_pv_cf["datetime"] = pd.to_datetime(
        rooftop_pv_cf["Unnamed: 0"].str.split("\'").str[1]
    )
    rooftop_pv_cf = (
        rooftop_pv_cf.set_index("datetime")
        .drop(columns=["Unnamed: 0"])
    )
    rooftop_pv_cf = rooftop_pv_cf / 1e3    
    rooftop_pv_cf = rooftop_pv_cf.sort_index()
    
    # Add county information
    rooftop_pv_cf.columns = rooftop_pv_cf_meta.gid
    gid_county_map = create_gid_county_map(dfcounty, rooftop_pv_cf_meta)
    rooftop_pv_cf = (
        rooftop_pv_cf.transpose()
        .merge(gid_county_map, left_index=True, right_index=True)
        .set_index('FIPS')
        .transpose()
    )
    
    rooftop_pv_cf.columns.name = ''
    rooftop_pv_cf.index.names = ['timestamp']
    rooftop_pv_cf.index = pd.to_datetime(rooftop_pv_cf.index)

    # Profiles start in UTC - convert to Central time
    rooftop_pv_cf = (
        rooftop_pv_cf.apply(lambda x: np.roll(x, shift=-6))
        .tz_convert(None)
        .tz_localize('Etc/GMT+6')
    )
    
    rooftop_pv_cf_2024 = rooftop_pv_cf.iloc[-24:].copy()
    rooftop_pv_cf_2024.index = rooftop_pv_cf_2024.index + pd.Timedelta(days=1)
    rooftop_pv_cf = pd.concat([rooftop_pv_cf, rooftop_pv_cf_2024])

    return rooftop_pv_cf

In [ ]:
dfcounty = gpd.read_file(
    os.path.join(
        'inputs',
        'shapefiles',
        'US_COUNTY_2022'
    )
) #TODO: file no longer in reeds repo

rooftop_pv_cf_meta = get_rooftop_pv_cf_meta()

rooftop_pv_cf_profiles_by_sector = {}
for sector in ['residential', 'commercial']:
    rooftop_pv_cf_profiles_by_sector[sector] = get_rooftop_pv_cf_profile(
        sector,
        rooftop_pv_cf_meta,
        dfcounty
    )

rooftop_pv_cf_profiles = (
    pd.concat(rooftop_pv_cf_profiles_by_sector.values(), axis=1)
    .groupby(level=0, axis=1)
    .mean()
)
weather_years = list(range(2007, 2014)) + list(range(2016, 2024))
rooftop_pv_cf_profiles = rooftop_pv_cf_profiles.loc[(
    rooftop_pv_cf_profiles.index.year.isin(weather_years)
)]

In [ ]:
reeds.io.write_profile_to_h5(
    rooftop_pv_cf_profiles.rename_axis(index='datetime').sort_index(axis=1).astype(np.float32),
    'distpv-reference_county.h5',
    'inputs/variability/multi_year'
)